In [1]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

In [2]:
def dual_basis(P):
    P = np.array(P, dtype=float)
    n = P.shape[0]
    tol=1e-10

    R = np.zeros((n, 2 * n))
    R[:, :n] = P

    for i in range(n):
        R[i, n + i] = 1

    for c in range(n):
        pivot_row = c

        for i in range(c + 1, n):
            if abs(R[i, c]) > abs(R[pivot_row, c]):
                pivot_row = i

        if abs(R[pivot_row, c]) < tol:
            raise ValueError("Matrix is not invertible")

        R[[c, pivot_row]] = R[[pivot_row, c]]

        pivot = R[c, c]
        R[c] = R[c] / pivot

        for i in range(n):
            if i != c:
                factor = R[i, c]
                R[i] = R[i] - factor * R[c]

    return R[:, n:]

In [7]:
P = rng.normal(size=(5, 5))
P

array([[-1.00961818, -0.20917557, -0.15922501,  0.54084558,  0.21465912],
       [ 0.35537271, -0.65382861, -0.12961363,  0.78397547,  1.49343115],
       [-1.25906553,  1.51392377,  1.34587542,  0.7813114 ,  0.26445563],
       [-0.31392281,  1.45802068,  1.96025832,  1.80163487,  1.31510376],
       [ 0.35738041, -1.20831863, -0.00445413,  0.65647494, -1.28836146]])

In [8]:
dual_basis(P)

array([[ 0.59966445, -1.24494784, -2.07238709,  1.38845448, -0.35130914],
       [ 2.89406599, -3.46599566, -4.43783509,  3.04980306, -1.33331462],
       [-4.35436018,  4.19166935,  6.05033725, -3.71684814,  1.58128852],
       [ 3.16995665, -2.97597432, -4.34126015,  3.03972355, -0.70979416],
       [-0.91764119,  1.37444685,  1.3542867 , -0.91346242,  0.00971199]])

In [9]:
print(dual_basis(P) @ P)

[[ 1.00000000e+00 -2.62081246e-16 -4.43182243e-16 -1.56949707e-16
  -2.02782797e-16]
 [ 4.80566181e-18  1.00000000e+00  3.01996750e-16  3.61435524e-16
  -6.26358983e-16]
 [ 7.15415473e-16 -8.51383348e-16  1.00000000e+00 -4.66412852e-16
  -5.57842577e-16]
 [-7.44598993e-16  5.78924721e-16  5.11850415e-16  1.00000000e+00
  -9.02737828e-18]
 [ 2.10616079e-16 -1.81949718e-16 -2.20515610e-16 -3.55738871e-17
   1.00000000e+00]]


In [10]:
print(np.allclose(dual_basis(P) @ P, np.eye(5)))

True


the matrix of all values ribj is the identity

In [11]:
x = rng.normal(size=5)
x

array([ 0.39512206,  0.42986369,  0.69604272, -1.18411797, -0.66170257])

In [12]:
coordinates = dual_basis(P) @ x
x2 = P @ coordinates
x2

array([ 0.39512206,  0.42986369,  0.69604272, -1.18411797, -0.66170257])

In [13]:
print(np.allclose(x,x2))

True


 applying the rows to a vector x gives its coordinates

##### Why are the Lagrange polynomial coefficients the columns of \(V^{-1}\)?

Each Lagrange polynomial equals 1 at its own node and 0 at all other nodes. For example, for the nodes \(-1,0,1\), the first polynomial must satisfy:

\[
L_0(-1)=1,\quad L_0(0)=0,\quad L_0(1)=0
\]

The Vandermonde matrix \(V\) takes the coefficients of a polynomial and gives its values at these nodes.

If \(c_j\) is the coefficient vector of \(L_j\), then:

\[
Vc_j=e_j
\]

where \(e_j\) is a vector containing 1 at position \(j\) and 0 elsewhere.

Multiplying both sides by \(V^{-1}\), we get:

\[
c_j=V^{-1}e_j
\]

Since multiplying a matrix by \(e_j\) selects its \(j\)-th column, each column of \(V^{-1}\) contains the coefficients of one Lagrange polynomial.

In [14]:
nodes = np.array([-1, 0, 1])
n = len(nodes)
V = np.zeros((n, n))
V

array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]])

In [15]:
for i in range(n):
    for j in range(n):
        V[i, j] = nodes[i] ** j
V

array([[ 1., -1.,  1.],
       [ 1.,  0.,  0.],
       [ 1.,  1.,  1.]])

In [16]:
L = dual_basis(V)
L

array([[ 0. ,  1. ,  0. ],
       [-0.5,  0. ,  0.5],
       [ 0.5, -1. ,  0.5]])

In [17]:
np.allclose(V @ L, np.eye(n))

True

It matches with Example 8.9